In [1]:
from custom_components.discrete_state_forecaster.model.temporal.time_key import (
    TimeKey,
)
from custom_components.discrete_state_forecaster.model.temporal.temporal_feature import (
    TemporalFeature,
)
from custom_components.discrete_state_forecaster.model.statistics.state_stats import (
    StateStats,
)

from custom_components.discrete_state_forecaster.model.statistics.distribution_stats import (
    DistributionStats,
)

from custom_components.discrete_state_forecaster.model.statistics.keyed_distribution_store import (
    KeyedDistributionStore,
)

from custom_components.discrete_state_forecaster.model.statistics.hierarchical_state_stats import (
    HierarchicalStateStats,
    PredictionResult,
)
from custom_components.discrete_state_forecaster.model.statistics.aggregated_stats import (
    AggregatedStats,
)

min_support = 5.0
ds = KeyedDistributionStore()
hss = HierarchicalStateStats(ds, min_support)

# hss.update(TimeKey.GLOBAL, "on", 10.0)
# hss.update(TimeKey.GLOBAL, "off", 10.0)

monday = TimeKey.from_temporal_feature(TemporalFeature("day", "monday"))
tuesday = TimeKey.from_temporal_feature(TemporalFeature("day", "tuesday"))


def distr(tk: TimeKey, a: PredictionResult | None) -> None:
    print(
        f"{str(tk).ljust(20, " ")} -> {str(a.distribution.distribution()) if a else None}, "
        f"confident={a.distribution.is_confident(min_support) if a else None}, "
        f"key={a.key if a else None}, "
        f"contributions={[ f'(key={c.key}, weight={c.weight}, support={c.support})' for c in a.contributions ] if a else None}"
    )


distr(TimeKey.GLOBAL, hss.predict(TimeKey.GLOBAL))
distr(monday, hss.predict(monday))
distr(tuesday, hss.predict(tuesday))
print("update monday is on for 10.0 seconds")
hss.update(monday, "on", 10.0)


distr(TimeKey.GLOBAL, hss.predict(TimeKey.GLOBAL))
distr(monday, hss.predict(monday))
distr(tuesday, hss.predict(tuesday))

print("update monday is on for 10.0 seconds")
hss.update(monday, "on", 10.0)

distr(TimeKey.GLOBAL, hss.predict(TimeKey.GLOBAL))
distr(monday, hss.predict(monday))
distr(tuesday, hss.predict(tuesday))

print("update tuesday is off for 20.0 seconds")
hss.update(tuesday, "off", 20.0)

distr(TimeKey.GLOBAL, hss.predict(TimeKey.GLOBAL))
distr(monday, hss.predict(monday))
distr(tuesday, hss.predict(tuesday))

print("update tuesday is on for 40.0 seconds")
hss.update(tuesday, "on", 40.0)

distr(TimeKey.GLOBAL, hss.predict(TimeKey.GLOBAL))
distr(monday, hss.predict(monday))
distr(tuesday, hss.predict(tuesday))

# ds.apply_decay(0.000001)
# print(ds.get_distribution(TimeKey.GLOBAL)._states)
# print(ds.get_distribution(monday)._states)
# print(ds.get_distribution(tuesday)._states)

# distr(TimeKey.GLOBAL, hss.predict(TimeKey.GLOBAL))
# distr(monday, hss.predict(monday))
# distr(tuesday, hss.predict(tuesday))

GLOBAL               -> None, confident=None, key=None, contributions=None
day = monday         -> None, confident=None, key=None, contributions=None
day = tuesday        -> None, confident=None, key=None, contributions=None
update monday is on for 10.0 seconds
GLOBAL               -> {'on': 1.0}, confident=True, key=GLOBAL, contributions=['(key=GLOBAL, weight=1.0, support=10.0)']
day = monday         -> {'on': 1.0}, confident=True, key=day = monday, contributions=['(key=day = monday, weight=1.0, support=10.0)']
day = tuesday        -> {'on': 1.0}, confident=True, key=day = tuesday, contributions=['(key=GLOBAL, weight=1.0, support=10.0)']
update monday is on for 10.0 seconds
GLOBAL               -> {'on': 1.0}, confident=True, key=GLOBAL, contributions=['(key=GLOBAL, weight=1.0, support=20.0)']
day = monday         -> {'on': 1.0}, confident=True, key=day = monday, contributions=['(key=day = monday, weight=1.0, support=20.0)']
day = tuesday        -> {'on': 1.0}, confident=True, key=day

In [2]:
day = TimeKey.from_temporal_feature(TemporalFeature("day", "monday"))
hour = TimeKey.from_temporal_feature(TemporalFeature("hour", 12))
list((day + hour).hierarchy())

[day = monday, hour = 12, day = monday, GLOBAL]

In [1]:
from custom_components.discrete_state_forecaster.model.learning.rolling_baseline import (
    RollingBaseline,
)
from custom_components.discrete_state_forecaster.model.statistics.distribution_stats import (
    DistributionStats,
)

In [2]:
rb = RollingBaseline(100.0, 10.0)
ds1 = DistributionStats()
ds1.update("on", 100.0)

ds2 = DistributionStats()
ds2.update("off", 100.0)
rb.update(ds1, 10.0)
rb.update(ds2, 10.0)

print("is_ready:", rb.is_ready())
print("current:", rb.current().distribution())

decay:  0.9048374180359595


ValueError: decay factor must be in (0, 1]. Got: -9.0